# Step 0: Import requisite libraries

In [445]:
import math
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn import preprocessing, ensemble, tree
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import matplotlib as plt
import seaborn

remove_excess = False

# Step 1: Import and cleaning of data

In [446]:
df_raw = pd.read_csv("Baza_customer_Telecom_v2.csv")

# Reducing number of negative samples
if remove_excess:
    df_pos = df_raw[df_raw["CHURN"] == "Yes"]
    df_neg = df_raw[df_raw["CHURN"] == "No"]

    neg_pos_ratio = 7/3
    neg_needed = int(len(df_pos) * neg_pos_ratio)

    df_nsampled = df_neg.sample(n=neg_needed, random_state=42)
    df = pd.concat([df_pos, df_nsampled]).sample(frac=1, random_state=42).reset_index(drop=True)
else:
    df = df_raw
df

,PID,CRM_PID_Value_Segment,EffectiveSegment,Billing_ZIP,KA_name,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,CHURN
0,123759242,Bronze,SOHO,6000.0,VM,2,NaN,NaN,2,40.17,0.0,40.17,NaN,No
1,126145737,Bronze,SOHO,6400.0,VM,3,NaN,NaN,3,40.17,0.0,40.17,13.39,No
2,123506355,Bronze,SOHO,6000.0,DI,2,3.0,NaN,5,40.17,0.0,40.17,20.09,No
3,112595585,Bronze,SOHO,4400.0,MT,1,2.0,NaN,3,40.17,0.0,40.17,40.17,No
4,115097935,Iron,SOHO,4000.0,AD,2,1.0,NaN,3,40.17,0.0,40.17,20.09,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8448,115901902,Iron,SOHO,4000.0,VU,2,1.0,NaN,3,56.50,0.0,56.50,28.25,No
8449,115800273,Bronze,SOHO,4006.0,VU,5,NaN,NaN,5,56.50,0.0,56.50,11.30,No
8450,160049889,Bronze,SOHO,4003.0,VU,3,1.0,NaN,4,56.50,0.0,56.50,18.83,No
8451,160095203,Bronze,SOHO,4023.0,VU,3,NaN,NaN,3,56.50,0.0,56.50,18.83,Yes


In [447]:
print(f"Length of dataset before cleaning: {len(df)}")
# Cleaning for PID
df = df.loc[df["PID"].notnull() & df["PID"].str.fullmatch(r"[0-9]*(SMB)?")] # Some PIDs were scientific notation (invalid); valid format is integer optionally followed by "SMB"

# Cleaning for value segment
df = df.loc[df["CRM_PID_Value_Segment"].notnull()]
df["CRM_PID_Value_Segment"] = df["CRM_PID_Value_Segment"].replace("Sliver", "Silver") # Fix assumed typo in one entry

# Cleaning for effective segment
df = df.loc[df["EffectiveSegment"].notnull()]

# Cleaning for ZIP
df = df.loc[df["Billing_ZIP"] % 1 == 0] # Remove non-integer ZIP codes (invalid; includes nulls)

# Cleaning for KA_name
df = df.loc[df["KA_name"].notnull()]
df["KA_name"] = df["KA_name"].replace("AD?", "AD") # Grouped AD? with AD under assumption that unsure attribution is correct

# Cleaning for active_subscribers
df = df.loc[df["Active_subscribers"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for not_active_subscribers
df["Not_Active_subscribers"] = df["Not_Active_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none inactive; minimum non-NaN value is 1)

# Cleaning for suspended_subscribers
df["Suspended_subscribers"] = df["Suspended_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none suspended; minimum non-NaN value is 1)

# Cleaning for total_SUBs
df = df.loc[df["Total_SUBs"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for AvgMobileRevenue
df = df.loc[df["AvgMobileRevenue "].notnull()] # Redundant (no nulls) but included for consistency
df = df.rename(columns={"AvgMobileRevenue ": "AvgMobileRevenue"}) # Remove trailing space from column name

# Cleaning for AvgFIXRevenue
df = df.loc[df["AvgFIXRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for TotalRevenue
df = df.loc[df["TotalRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for ARPU
df = df.loc[df["ARPU"].notnull()]

# Cleaning for CHURN
df = df.loc[df["CHURN"].isin(["No", "Yes"])] # Remove invalid CHURN values (only valid values are "No" and "Yes"); redundant (no invalids) but included for consistency

print(f"Length of dataset after cleaning: {len(df)}") # Small length change, hence valid to discard nulls/invalid entries

# Source - https://stackoverflow.com/a/48035423
# Posted by BENY, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-02, License - CC BY-SA 4.0

df.apply(lambda x : pd.factorize(x)[0]).corr(method='pearson', min_periods=1)
df.groupby("CRM_PID_Value_Segment")["TotalRevenue"].mean().sort_values()

Length of dataset before cleaning: 8453
Length of dataset after cleaning: 8419


CRM_PID_Value_Segment
Iron         59.888252
Bronze       79.287700
Silver      146.561462
SE          166.550278
Lead        170.716667
Gold        239.062784
SME         308.828669
Platinum    360.134227
Name: TotalRevenue, dtype: float64

# Step 2: Data Preprocessing

In [448]:
X = df.drop(columns=["CHURN"])
y = df["CHURN"].map({"No": 0, "Yes": 1}) # Convert CHURN to binary values (0 for No, 1 for Yes)

X["PID_is_SMB"] = X["PID"].str.endswith("SMB").astype("string") # Create binary feature indicating if PID ends with "SMB"
X["PID"]=X["PID"].str.replace("SMB", "").astype(int) # Remove "SMB" from PID and convert to integer

# Definition of categorical and numerical features for preprocessing
cat_features = ["CRM_PID_Value_Segment", "EffectiveSegment", "KA_name", "PID_is_SMB"]
num_features = ["Billing_ZIP", "Active_subscribers", "Not_Active_subscribers", "Suspended_subscribers", "Total_SUBs", "AvgMobileRevenue", "AvgFIXRevenue", "TotalRevenue", "ARPU"]

# Selection of features for model training

train_features = cat_features + num_features

class_weights = {0: len(y == 1), 1: len(y == 0)}

# train_features = [
#     "PID_is_SMB",
#     "CRM_PID_Value_Segment", 
#     "Active_subscribers",
#     "AvgMobileRevenue",
#     "TotalRevenue"
#     ]

X = X[train_features]

# Definition of ColumnTransformer for preprocessing
# c_transformer = ColumnTransformer(
#     transformers=[
#         ("categorical", preprocessing.OneHotEncoder(handle_unknown="ignore"), list(set(cat_features) & set(train_features)) ),
#         ("numerical", "passthrough", list(set(num_features) & set(train_features)) )
#     ]
# )

# print(f"Column transformer has categorical features {list(set(cat_features) & set(train_features))} and numerical features {list(set(num_features) & set(train_features))}")


# XGBoost prep
for col in cat_features:
    X[col] = X[col].astype('category')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Model pipeline
clf_p = Pipeline([
    # ("preprocessor", c_transformer),
    # ("classifier", ensemble.RandomForestClassifier(
    #     n_estimators=500,
    #     class_weight="balanced",
    #     n_jobs=-1,
    #     random_state=42))

    # ("classifier", tree.DecisionTreeClassifier(
    #     class_weight="balanced",
    #     splitter="best",
    #     max_depth=None,
    #     random_state=42))

    ("classifier", xgb.XGBClassifier(
        enable_categorical = True,
        tree_method="hist",
        n_estimators = 10000,
        learning_rate = 0.01,
        max_depth = 6,
        early_stopping_rounds = 50,
        objective = 'binary:logistic',
        eval_metric = 'aucpr',
        scale_pos_weight = len(df[df["CHURN"] == "No"]) / len(df[df["CHURN"] == "Yes"]),
        max_delta_step = 1,
        min_child_weight = 5,
        random_state = 42
    ))
])


# Fit model
clf_p.fit(X_train, y_train, classifier__eval_set = [(X_test, y_test)], classifier__verbose = False)

# Predict on test set
y_pred_p = clf_p.predict_proba(X_test)
t_df = pd.DataFrame(X_test)
t_df["Predicted_Churn_Probability"] = y_pred_p[:, 1] # Get probabilities for positive class (CHURN = Yes)
t_df["Predicted_Churn"] = (t_df["Predicted_Churn_Probability"] >= 0.5).astype(int) # Convert probabilities to binary predictions using threshold of 0.5
t_df["Actual_Churn"] = y_test.values # Add actual CHURN values for comparison
t_df["Prediction_type"] = np.select(
    [
        (t_df["Predicted_Churn"] == 1) & (t_df["Actual_Churn"] == 1),
        (t_df["Predicted_Churn"] == 0) & (t_df["Actual_Churn"] == 0),
        (t_df["Predicted_Churn"] == 1) & (t_df["Actual_Churn"] == 0),
        (t_df["Predicted_Churn"] == 0) & (t_df["Actual_Churn"] == 1)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

# Evaluate model performance
threshold = 0.5
y_pred = y_pred_p[:, 1] >= threshold
print(classification_report(y_test, y_pred))
print("Test confusion matrix with threshold", (str(threshold) + ":\n"), confusion_matrix(y_test, y_pred))
print("Training confusion matrix (to check for overfit) with threshold", (str(threshold) + ":\n"),confusion_matrix(y_train, clf_p.predict(X_train)))
t_df.to_csv("Predicted_Churn_Results.csv", index=False) # Save predictions to CSV for further analysis
t_df[t_df["Prediction_type"] == "True Positive"].sort_values(by="Predicted_Churn_Probability", ascending=False).head(20) # Sort by predicted probability for easier analysis


              precision    recall  f1-score   support

           0       0.94      0.61      0.74      1575
           1       0.07      0.45      0.13       109

    accuracy                           0.60      1684
   macro avg       0.51      0.53      0.43      1684
weighted avg       0.88      0.60      0.70      1684

Test confusion matrix with threshold 0.5:
 [[957 618]
 [ 60  49]]
Training confusion matrix (to check for overfit) with threshold 0.5:
 [[3916 2385]
 [ 114  320]]


,CRM_PID_Value_Segment,EffectiveSegment,KA_name,PID_is_SMB,Billing_ZIP,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,Predicted_Churn_Probability,Predicted_Churn,Actual_Churn,Prediction_type
4543,Gold,VSE,Daniela Stefanova,False,4023.0,29,0.0,0.0,29,297.33,0.0,297.33,10.25,0.547122,1,1,True Positive
4919,Platinum,VSE,Ginka Vachkova,False,4210.0,20,2.0,1.0,23,381.00,0.0,381.00,19.05,0.547122,1,1,True Positive
4997,Platinum,VSE,Daniela Stefanova,False,4199.0,27,0.0,0.0,27,403.33,0.0,403.33,14.94,0.547122,1,1,True Positive
4446,Gold,VSE,RJ,False,5800.0,28,1.0,0.0,29,284.33,0.0,284.33,10.15,0.543362,1,1,True Positive
4664,Gold,VSE,Daniela Stefanova,False,4002.0,15,1.0,0.0,16,316.33,0.0,316.33,21.09,0.537308,1,1,True Positive
4390,Gold,VSE,Tatiana Trifonova,False,4002.0,18,2.0,1.0,21,276.67,0.0,276.67,15.37,0.537308,1,1,True Positive
5148,Platinum,VSE,VT,False,4363.0,30,0.0,0.0,30,487.83,0.0,487.83,16.26,0.533442,1,1,True Positive
3436,Platinum,SOHO,DI,False,6000.0,1,20.0,0.0,21,181.17,0.0,181.17,181.17,0.532334,1,1,True Positive
8403,Bronze,SOHO,Daniela Stefanova,False,4000.0,2,4.0,0.0,6,55.33,0.0,55.33,27.67,0.531979,1,1,True Positive
8423,Bronze,SOHO,Daniela Stefanova,False,4000.0,1,6.0,0.0,7,55.83,0.0,55.83,55.83,0.524075,1,1,True Positive


In [449]:
t_df[t_df["Prediction_type"] == "False Positive"].sort_values(by="Predicted_Churn_Probability", ascending=False).head(20) # Sort by predicted probability for easier analysis

,CRM_PID_Value_Segment,EffectiveSegment,KA_name,PID_is_SMB,Billing_ZIP,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,Predicted_Churn_Probability,Predicted_Churn,Actual_Churn,Prediction_type
7312,SME,SME,Ginka Vachkova,False,4003.0,25,0.0,0.0,25,487.33,0.0,487.33,19.49,0.547122,1,0,False Positive
5106,Platinum,VSE,Daniela Stefanova,False,4000.0,21,7.0,0.0,28,460.17,0.0,460.17,21.91,0.547122,1,0,False Positive
7560,Platinum,VSE,Ginka Vachkova,False,4300.0,23,2.0,0.0,25,347.33,0.0,347.33,15.10,0.547122,1,0,False Positive
7457,SME,SME,Daniela Stefanova,False,4288.0,26,4.0,0.0,30,366.33,0.0,366.33,14.09,0.547122,1,0,False Positive
7559,Platinum,VSE,Ginka Vachkova,False,4023.0,19,5.0,0.0,24,390.83,0.0,390.83,20.57,0.547122,1,0,False Positive
7696,SME,SME,Tatiana Trifonova,False,4150.0,20,1.0,0.0,21,144.17,0.0,144.17,7.21,0.547122,1,0,False Positive
7711,Platinum,VSE,Tatiana Trifonova,False,4300.0,19,2.0,1.0,22,271.00,0.0,271.00,14.26,0.547122,1,0,False Positive
7547,Gold,VSE,Ginka Vachkova,False,4021.0,22,6.0,1.0,29,490.17,0.0,490.17,22.28,0.547122,1,0,False Positive
7295,SME,SME,Ginka Vachkova,False,4400.0,25,1.0,0.0,26,396.17,0.0,396.17,15.85,0.547122,1,0,False Positive
7300,Gold,VSE,Ginka Vachkova,False,4000.0,28,0.0,1.0,29,280.33,0.0,280.33,10.01,0.547122,1,0,False Positive


In [450]:
# Testing per-group
test_segment = "Bronze"
segment_test = y_test[X_test["CRM_PID_Value_Segment"] == test_segment]
segment_pred = y_pred[X_test["CRM_PID_Value_Segment"] == test_segment]

print(classification_report(segment_test, segment_pred))

              precision    recall  f1-score   support

           0       0.94      0.76      0.84       703
           1       0.09      0.35      0.15        49

    accuracy                           0.74       752
   macro avg       0.52      0.56      0.50       752
weighted avg       0.89      0.74      0.80       752

